# Machine Learning Zoomcamp 2026 - Homework 4: Evaluation Metrics for Classification

- **Dataset:** `course_lead_scoring_2026.csv`
- **Target:** `converted`


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()


## Data Preparation and Splitting

- Categorical features filled with `'NA'`
- Numerical features filled with `0.0`
- Split with `random_state=1`


In [ ]:
target = 'converted'
features = df.drop(columns=[target]).copy()
categorical = features.select_dtypes(include=['object']).columns.tolist()
numerical = features.select_dtypes(exclude=['object']).columns.tolist()

features[categorical] = features[categorical].fillna('NA')
features[numerical] = features[numerical].fillna(0.0)
prepared_df = features.assign(**{target: df[target]})

df_full_train, df_test = train_test_split(prepared_df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.pop(target).values
y_val = df_val.pop(target).values
y_test = df_test.pop(target).values


## Question 1: ROC AUC Feature Importance


In [ ]:
candidates = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']
aucs = {}
for col in candidates:
    score = roc_auc_score(y_train, df_train[col])
    if score < 0.5:
        score = roc_auc_score(y_train, -df_train[col])
    aucs[col] = score
    print(f'AUC({col}) = {score:.4f}')

print('Q1 Answer (Highest AUC numerical variable):', max(candidates, key=aucs.get))


## Question 2: Model Training and Validation AUC


In [ ]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_probs)
print('Validation ROC AUC:', val_auc)
print('Q2 Answer (Rounded 3 digits):', round(val_auc, 3))


## Question 3: Precision and Recall Intersection


In [ ]:
thresholds = np.arange(0.0, 1.01, 0.01)
diffs = []
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    p = precision_score(y_val, preds, zero_division=0)
    r = recall_score(y_val, preds, zero_division=0)
    if p == 0 and r == 0:
        continue
    diffs.append((abs(p - r), round(t, 2), p, r))

diffs.sort(key=lambda x: x[0])
print('Q3 Answer (Intersection threshold):', diffs[0][1])


## Question 4: Maximal F1 Score Threshold


In [ ]:
f1_scores = []
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    score = f1_score(y_val, preds, zero_division=0)
    f1_scores.append((score, round(t, 2)))

f1_scores.sort(key=lambda x: -x[0])
print('Q4 Answer (Maximal F1 threshold):', f1_scores[0][1])


## Question 5: 5-Fold Cross-Validation Standard Deviation


In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=1)
X_full = df_full_train.drop(columns=[target])
y_full = df_full_train[target].values

cv_scores = []
for train_idx, val_idx in kfold.split(X_full):
    f_tr = X_full.iloc[train_idx]
    f_va = X_full.iloc[val_idx]
    f_y_tr = y_full[train_idx]
    f_y_va = y_full[val_idx]

    dv_f = DictVectorizer(sparse=False)
    X_f_tr = dv_f.fit_transform(f_tr.to_dict(orient='records'))
    X_f_va = dv_f.transform(f_va.to_dict(orient='records'))

    m_f = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
    m_f.fit(X_f_tr, f_y_tr)
    cv_scores.append(roc_auc_score(f_y_va, m_f.predict_proba(X_f_va)[:, 1]))

cv_std = np.std(cv_scores)
print('CV standard deviation:', cv_std)
print('Q5 Answer (Rounded std):', round(cv_std, 3))


## Question 6: Hyperparameter Tuning Across C


In [ ]:
c_candidates = [0.000001, 0.001, 1.0]
c_results = {}
for c in c_candidates:
    scores = []
    for train_idx, val_idx in kfold.split(X_full):
        f_tr = X_full.iloc[train_idx]
        f_va = X_full.iloc[val_idx]
        f_y_tr = y_full[train_idx]
        f_y_va = y_full[val_idx]

        dv_f = DictVectorizer(sparse=False)
        X_f_tr = dv_f.fit_transform(f_tr.to_dict(orient='records'))
        X_f_va = dv_f.transform(f_va.to_dict(orient='records'))

        m_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000)
        m_c.fit(X_f_tr, f_y_tr)
        scores.append(roc_auc_score(f_y_va, m_c.predict_proba(X_f_va)[:, 1]))

    c_results[c] = (round(np.mean(scores), 3), round(np.std(scores), 3))
    print(f'C={c}: mean={c_results[c][0]}, std={c_results[c][1]}')

best_c = max(c_candidates, key=lambda c: (c_results[c][0], -c_results[c][1], -c))
print('Q6 Answer (Best C):', best_c)
